In [1]:
from pathlib import Path

import pandas as pd
import rasterio
from pyproj import Transformer

import unicodedata
import re


In [2]:
ROOT = Path("..")      # Notebook liegt in notebooks/

RAW_DIR = ROOT / "data" / "raw"
PROCESSED_DIR = ROOT / "data" / "processed"

SWISSNAMES_PATH = RAW_DIR / "swissNAMES3D_PKT.csv"
ALTIREGIO_PATH = RAW_DIR / "swissaltiregio_2056_5728.tif"

OUTPUT_PATH = PROCESSED_DIR / "places_candidates.csv"

In [3]:
#Normalisieren der Ortsnamen
def make_place_id(name):
    name = str(name).lower()

    # Umlaute / Akzente entfernen: Zürich -> zurich, Gruyères -> gruyeres
    name = unicodedata.normalize("NFKD", name)
    name = "".join(c for c in name if not unicodedata.combining(c))

    # alles ausser Buchstaben/Zahlen durch _ ersetzen
    name = re.sub(r"[^a-z0-9]+", "_", name)

    # doppelte/führende/endende _ entfernen
    name = re.sub(r"_+", "_", name).strip("_")

    return name

In [4]:
cols = [
    "NAME",
    "STATUS",
    "SPRACHCODE",
    "OBJEKTART",
    "EINWOHNERKATEGORIE",
    "E",
    "N",
]

df = pd.read_csv(
    SWISSNAMES_PATH,
    sep=";",
    encoding="utf-8-sig",
    usecols=cols,
    low_memory=False
)

df.head()

,NAME,STATUS,SPRACHCODE,OBJEKTART,EINWOHNERKATEGORIE,E,N
0,Kleinandelfingen,offiziell,Hochdeutsch inkl. Lokalsprachen,Ausfahrt,k_W,2694037,1273365
1,Kleinandelfingen,offiziell,Hochdeutsch inkl. Lokalsprachen,Ausfahrt,k_W,2694225,1272961
2,Winterthur-Töss,offiziell,Hochdeutsch inkl. Lokalsprachen,Ausfahrt,k_W,2695211,1260579
3,Wülflingen,offiziell,Hochdeutsch inkl. Lokalsprachen,Ausfahrt,k_W,2694253,1263269
4,Wülflingen,offiziell,Hochdeutsch inkl. Lokalsprachen,Ausfahrt,k_W,2694494,1263592


In [5]:
#Datensatz filtern nach Ortschaften und offizieller Schreibweise (keine Duplikate)
places = df[
    (df["OBJEKTART"] == "Ort") &
    (df["STATUS"] == "offiziell")
].copy()

places.head()

,NAME,STATUS,SPRACHCODE,OBJEKTART,EINWOHNERKATEGORIE,E,N
329433,Collonge-Bellerive,offiziell,Franzoesisch inkl. Lokalsprachen,Ort,2'000 bis 9'999,2504410,1123024
329445,Widnau,offiziell,Hochdeutsch inkl. Lokalsprachen,Ort,2'000 bis 9'999,2765726,1252889
329452,Dornach,offiziell,Hochdeutsch inkl. Lokalsprachen,Ort,2'000 bis 9'999,2613442,1258650
329464,Herrliberg,offiziell,Hochdeutsch inkl. Lokalsprachen,Ort,2'000 bis 9'999,2689097,1237987
329470,Zuchwil,offiziell,Hochdeutsch inkl. Lokalsprachen,Ort,2'000 bis 9'999,2608894,1228027


In [6]:
#Zuweisen Sprachcode
language_region_map = {
    "Franzoesisch inkl. Lokalsprachen": "fr",
    "Hochdeutsch inkl. Lokalsprachen": "de",
    "Italienisch inkl. Lokalsprachen": "it",
    "Rumantsch Grischun inkl. Lokalsprachen": "rm",
    "mehrsprachig": "multi",
}

In [7]:
#Zuweisung der Einwohnerzahl-Klassen in Kategorien
einwohner_rank = {
    "< 20": 1,
    "20 bis 49": 2,
    "50 bis 99": 3,
    "100 bis 999": 4,
    "1'000 bis 1'999": 5,
    "2'000 bis 9'999": 6,
    "10'000 bis 49'999": 7,
    "50'000 bis 100'000": 8,
    "> 100'000": 9,
}

places["population_rank"] = places["EINWOHNERKATEGORIE"].map(einwohner_rank)

In [8]:
#Höhe aus AltiRegio samplen (Konsistenz mit Klimadaten-Aufbereitung
with rasterio.open(ALTIREGIO_PATH) as src:
    coords = list(zip(places["E"], places["N"]))
    places["elevation_m"] = [value[0] for value in src.sample(coords)]

In [9]:
#Koordinatensystem umrechnen für historische Klimadaten
transformer = Transformer.from_crs("EPSG:2056", "EPSG:4326", always_xy=True)

places["lon"], places["lat"] = transformer.transform(
    places["E"].to_numpy(),
    places["N"].to_numpy()
)

In [10]:
#Zonen definieren
def assign_zone(elevation):
    if elevation < 700:
        return 1
    elif elevation < 1500:
        return 2
    else:
        return 3

zone_names = {
    1: "Mittelland",
    2: "Voralpen",
    3: "Alpen",
}

places["zone_expected"] = places["elevation_m"].apply(assign_zone)
places["zone_label"] = places["zone_expected"].map(zone_names)

In [11]:
#Pro Höhenzone je 20 Ortschaften auswählen
selected = (
    places
    .sort_values(["zone_expected", "population_rank", "NAME"], ascending=[True, False, True])
    .groupby("zone_expected")
    .head(20)
    .copy()
)

In [12]:
#places.csv Schema anwenden
selected["place_id"] = selected["NAME"].apply(make_place_id)
selected["place_name"] = selected["NAME"]
selected["x_lv95"] = selected["E"]
selected["y_lv95"] = selected["N"]
selected["place_type"] = "Ort"
selected["is_featured"] = False
selected["short_description"] = ""
selected["marker_priority"] = 2
selected["zoom_level"] = 9
selected["display_order"] = range(1, len(selected) + 1)
selected["image_name"] = ""
selected["canton"] = ""
selected["language_region"] = selected["SPRACHCODE"].map(language_region_map)

In [13]:
#export places.csv
columns = [
    "place_id",
    "place_name",
    "zone_expected",
    "zone_label",
    "canton",
    "language_region",
    "lat",
    "lon",
    "x_lv95",
    "y_lv95",
    "place_type",
    "is_featured",
    "short_description",
    "elevation_m",
    "marker_priority",
    "zoom_level",
    "display_order",
    "image_name",
]

places_out = selected[columns].copy()

places_out.to_csv(
    OUTPUT_PATH,
    index=False,
    encoding="utf-8-sig"
)
places_out.head()

,place_id,place_name,zone_expected,zone_label,canton,language_region,lat,lon,x_lv95,y_lv95,place_type,is_featured,short_description,elevation_m,marker_priority,zoom_level,display_order,image_name
382886,basel,Basel,1,Mittelland,,de,47.557255,7.587586,2611212,1267404,Ort,False,,255.382416,2,9,1,
382890,bern,Bern,1,Mittelland,,de,46.948159,7.441614,2600227,1199675,Ort,False,,542.027283,2,9,2,
382897,geneve,Genève,1,Mittelland,,fr,46.208208,6.145830,2500226,1118241,Ort,False,,376.209351,2,9,3,
382894,lausanne,Lausanne,1,Mittelland,,fr,46.520554,6.634979,2538331,1152456,Ort,False,,490.114288,2,9,4,
382905,winterthur,Winterthur,1,Mittelland,,de,47.498744,8.727662,2697124,1261685,Ort,False,,440.636688,2,9,5,
